# Sweep post-processing on the date holdout

Run this notebook on Kaggle. Attach the competition data and the private
repo dataset (a ZIP of this repo without `data/`, `.env`, or `kaggle.json`).
Do not make the GitHub repository public.

CPU is enough for the classic baseline. Leave the accelerator on **None**
unless U-Net weights are attached and you also want the U-Net rows. A GPU is
not required for the baseline.

Run every cell from the top. This notebook does not submit to Kaggle.
It does not set a Hugging Face token. It does not read the test images,
and it does not put MAGFiLO in the repo.

Send back the printed table and `/kaggle/working/postprocess_sweep.csv`.


## What is scored

The date holdout is `filament.train_data.split_training_images` (the most
recent 20% of calendar dates). A whole date stays on one side.

Ground truth is `filament.annotations.annotator_views` on each holdout
file. That is one tuple per original annotator entry. It is not
`TrainImage.annotations` (the keep-one-mask list) and it is not the
semantic union. The same prediction is scored against each view.

The score adds `filament.metrics.fast_view_counts` across those views
and reads `PqCounts.pq`. Predictions are a disjoint label image, so
this notebook uses `fast_view_counts`, not `reference_view_counts`.
A zero denominator is NaN. Per-image PQ is not averaged. This notebook
does not reimplement the ratio.

`min_area` is swept over `(80, 140, 240, 400, 700, 1200, 2000, 3000)`.
`close_size` is `0` (off) and `3` (3×3 gap kernel). The areas are
roughly log-spaced from 80 to 3000, and 240 and 400 stay in the grid.
The 2026-11-01 run of this grid is in `docs/RESULTS.md` and
`results/postprocess_sweep_2026-11-01.csv`.

On a baseline-only holdout of 142 files and 231 annotator views,
`close_size` 0 scored higher than 3. The best baseline row was `min_area`
1200 and `close_size` 0 (pooled PQ 0.0734). Closing lowered the baseline.
The 12-epoch U-Net peaked at `min_area` 400 (pooled PQ 0.2886) and fell
after that (0.2783 at 700, 0.2501 at 1200). The U-Net rows are the same
for `close_size` 0 and 3. The baseline and the U-Net peak at different
areas, and the library default of 400 follows the U-Net peak. The grid is
one holdout, so the chosen `min_area` stays **[PENDIENTE]**.


## 1. `src` on the path

`src` is the directory that contains `filament/train_data.py`. The walk
matches the inference notebook: `PYTHONPATH`, then `sys.path`, then any
depth under `/kaggle/input`, skipping every `competitions` directory.
A local checkout is used when it is already the working directory.

`WEIGHTS_PATH` in the next cell defaults to `None`. Set it to a `.pt`
file to load that checkpoint. The loaded path is printed. `None` searches
every `*.pt` under `/kaggle/working` and under `/kaggle/input`, skipping
the `competitions` tree. One file is used, so `unet_12ep.pt` next to the
repo zip is loaded when it is the only `.pt`. Several files use `unet.pt`
when exactly one candidate has that name. Several files and no single
`unet.pt` stop and list every candidate: set `WEIGHTS_PATH`. No file
prints a warning and scores the classic baseline only.

Kaggle extracts an uploaded `.pt` because `torch.save` writes a zip.
`unet.pt` becomes a directory that contains `unet/data.pkl`. That
directory is not a checkpoint. The loader re-zips it to a real file
under `/kaggle/working` and prints that path. If the folder is not one
torch archive, the notebook stops and tells you to set `WEIGHTS_PATH`
to a real `.pt` file.

The notebook prints a fingerprint of the checkpoint it loads: the resolved path, the file size in bytes, the MD5 of those file bytes, epochs, encoder, and image size from the checkpoint, and a hash of the `state_dict` tensor bytes. A field that is absent is printed as `not stored`. The file MD5 of a re-zip is not comparable to the original file MD5. The tensor hash is the one to compare. The same fingerprint is printed for every other `.pt` file and every extracted torch archive (for example `unet/data.pkl`) under `/kaggle/input` outside `competitions`.


In [ ]:
import os
import sys
import zipfile
from pathlib import Path

# Set this to a .pt path to load that checkpoint.
# None searches /kaggle/working and /kaggle/input, skipping competitions.
# A directory from Kaggle extracting a .pt zip is re-zipped under /kaggle/working.
WEIGHTS_PATH = None

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")


def _has_train_data(directory: Path) -> bool:
    return (directory / "filament" / "train_data.py").is_file()


def _src_from_entries(entries: list[str]) -> Path | None:
    for part in entries:
        if part == "":
            continue
        directory = Path(part)
        if _has_train_data(directory):
            return directory
    return None


def _local_src() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if _has_train_data(src):
            return src
    return None


def _walk_input(root: Path):
    """Yield `(directory, filenames)` under `root`, skipping `competitions`."""
    if not root.is_dir():
        return
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [name for name in sorted(dirnames) if name != "competitions"]
        yield Path(dirpath), filenames


def _find_src() -> Path:
    found = _local_src()
    if found is None:
        raw = os.environ.get("PYTHONPATH", "")
        found = _src_from_entries(raw.split(os.pathsep))
    if found is None:
        found = _src_from_entries(list(sys.path))
    if found is None:
        for directory, _filenames in _walk_input(KAGGLE_INPUT):
            if _has_train_data(directory):
                found = directory
                break
    if found is None:
        raise FileNotFoundError(
            "Could not find the directory that contains "
            "filament/train_data.py. Put that src directory on PYTHONPATH, "
            "or attach the private repo under /kaggle/input outside "
            "competitions."
        )
    return found


def _pt_under(root: Path) -> list[Path]:
    """Every ``*.pt`` file or directory under ``root``.

    A directory counts because Kaggle extracts an uploaded ``.pt`` zip
    into a folder of that name. ``competitions`` is not walked.
    """
    found: list[Path] = []
    if not root.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            if name.endswith(".pt"):
                found.append(directory / name)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
    return found


def _weight_candidates() -> list[Path]:
    """``.pt`` files and extracted archives, working then input."""
    return _pt_under(KAGGLE_WORKING) + _pt_under(KAGGLE_INPUT)


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )


WEIGHTS_SOURCE: Path | None = None


def _resolve_weights(explicit: Path | str | None) -> Path | None:
    """Return the checkpoint ``explicit`` names, or one discovered ``*.pt``.

    One discovered file or extracted directory is used, including
    ``unet_12ep.pt`` when it is the only ``.pt``. Several matches use
    ``unet.pt`` when exactly one candidate has that name. Several matches
    and no single ``unet.pt`` raise and list every candidate. An extracted
    torch archive is re-zipped under ``/kaggle/working`` and that file is
    returned. No match returns ``None``.
    """
    global WEIGHTS_SOURCE
    WEIGHTS_SOURCE = None
    if explicit is not None:
        path = Path(explicit)
        if not path.exists():
            raise FileNotFoundError(
                f"WEIGHTS_PATH is set to {path}, but that file does not exist. "
                "Set WEIGHTS_PATH to an existing .pt file."
            )
        WEIGHTS_SOURCE = path
        path = _materialize_checkpoint(path)
        print(f"loaded weights={path}")
        return path
    candidates = _weight_candidates()
    if not candidates:
        return None
    if len(candidates) == 1:
        WEIGHTS_SOURCE = candidates[0]
        chosen = _materialize_checkpoint(candidates[0])
        print(f"loaded weights={chosen}")
        return chosen
    named = [path for path in candidates if path.name == "unet.pt"]
    if len(named) == 1:
        WEIGHTS_SOURCE = named[0]
        chosen = _materialize_checkpoint(named[0])
        print(f"Several .pt files were found. Using the one named unet.pt: {chosen}")
        print(f"loaded weights={chosen}")
        return chosen
    listed = "\n".join(f"  {path}" for path in candidates)
    if named:
        detail = "Several files named unet.pt were found."
    else:
        detail = "Several .pt files were found and none is named unet.pt."
    raise RuntimeError(f"{detail} Set WEIGHTS_PATH to the file to load.\n{listed}")


def _is_archive_folder(directory: Path) -> bool:
    """True when this directory is one extracted torch archive."""
    if not directory.is_dir():
        return False
    try:
        names = {child.name for child in directory.iterdir()}
    except OSError:
        return False
    data_pkl = directory / "data.pkl"
    if "data.pkl" not in names or not data_pkl.is_file() or data_pkl.is_symlink():
        return False
    return "version" in names or "byteorder" in names or (directory / "data").is_dir()


def _input_checkpoints() -> list[Path]:
    """``.pt`` files and extracted archives under input, skipping competitions.

    An extracted archive may be a directory named ``unet.pt`` or a folder
    such as ``unet`` that contains ``data.pkl``.
    """
    found: list[Path] = []
    if not KAGGLE_INPUT.is_dir():
        return found

    def _walk_error(error: OSError) -> None:
        print(f"fingerprint other error={error}")

    for dirpath, dirnames, filenames in os.walk(KAGGLE_INPUT, onerror=_walk_error):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            child = directory / name
            if name.endswith(".pt"):
                found.append(child)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
        if _is_archive_folder(directory) and not directory.parent.name.endswith(".pt"):
            found.append(directory)
    return found


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


SRC = _find_src()
_put_on_path(SRC)
print(f"src={SRC}")
WEIGHTS_PATH = _resolve_weights(WEIGHTS_PATH)
if WEIGHTS_PATH is None:
    print(
        "WARNING: no U-Net weights were found. "
        "WEIGHTS_PATH is None, and no .pt file is under /kaggle/working "
        "or under /kaggle/input outside competitions. "
        "Only the classic baseline will be scored."
    )

## 2. Date holdout, one view per annotator entry

Train images and the train JSON are the competition paths in
`filament.annotations`. Test images are not opened.


In [ ]:
import cv2
import numpy as np

from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    TrainImage,
    annotator_views,
    load_train_annotations,
)
from filament.train_data import split_training_images

loaded = load_train_annotations(TRAIN_ANNOTATIONS_PATH)
split = split_training_images(loaded.images)
HOLDOUT = split.holdout
if not HOLDOUT:
    raise RuntimeError("The date holdout has no images.")
VIEW_COUNT = sum(len(annotator_views(image)) for image in HOLDOUT)
print(
    f"holdout files={len(HOLDOUT)} annotator views={VIEW_COUNT} "
    f"train files={len(split.train)}"
)
sample = HOLDOUT[0]
print(f"frame={sample.height}x{sample.width}")
print(f"images={TRAIN_IMAGES_DIR}")
print("test images are not read")

## 3. Classic baseline

`segment_filaments` is called with each `min_area` and `close_size`.
`block_size` and `offset` stay at their defaults. There is no opening.

Counts come from `filament.metrics.fast_view_counts`. This cell adds
the `PqCounts` and reads `.pq`. It does not define the metric.


In [ ]:
import csv
import math
from collections.abc import Callable, Iterator

from filament.baseline import segment_filaments
from filament.metrics import PqCounts, fast_view_counts

MIN_AREAS = (80, 140, 240, 400, 700, 1200, 2000, 3000)
CLOSE_SIZES = (0, 3)
OUT_PATH = Path("/kaggle/working/postprocess_sweep.csv")
COLUMNS = (
    "predictor",
    "min_area",
    "close_size",
    "pooled_pq",
    "holdout_files",
    "annotator_views",
)


def _read_gray(image: TrainImage) -> np.ndarray:
    path = Path(TRAIN_IMAGES_DIR) / image.file_name
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(path)
    expected = (image.height, image.width)
    if gray.shape != expected or gray.dtype != np.uint8:
        raise ValueError(
            f"{image.file_name} is {getattr(gray, 'shape', None)} "
            f"{getattr(gray, 'dtype', None)}, expected {expected} uint8"
        )
    return np.ascontiguousarray(gray)


def _view_masks(image: TrainImage, view: tuple) -> list[np.ndarray]:
    """One mask per annotation in this annotator view. Not a union.

    Empty rasterizations are omitted. ``category_id`` is not read.
    ``semantic_only`` is not drawn. ``image.annotations`` is not read.
    """
    expected = (image.height, image.width)
    masks: list[np.ndarray] = []
    for annotation in view:
        mask = np.asarray(annotation.rasterize(image.height, image.width))
        if mask.shape != expected:
            raise ValueError(
                f"{image.file_name} ground truth has shape {mask.shape}, "
                f"expected {expected}"
            )
        if np.any(mask):
            masks.append(np.ascontiguousarray(mask))
    return masks


def _iter_views(
    make_labels: Callable[[np.ndarray], np.ndarray],
) -> Iterator[tuple[np.ndarray, list[np.ndarray]]]:
    """Yield `(label_image, annotator masks)` one holdout file at a time."""
    total = len(HOLDOUT)
    for index, image in enumerate(HOLDOUT, start=1):
        gray = _read_gray(image)
        labels = make_labels(gray)
        if labels.shape != gray.shape:
            raise ValueError(
                f"{image.file_name} labels {labels.shape} do not match {gray.shape}"
            )
        for view in annotator_views(image):
            yield labels, _view_masks(image, view)
        del gray, labels
        if index == 1 or index % 20 == 0 or index == total:
            print(f"  {index}/{total} {image.file_name}", flush=True)


def _format_pq(value: float) -> str:
    number = float(value)
    if math.isnan(number):
        return "nan"
    return f"{number:.6f}"


def _write_rows(rows: list[dict[str, str]]) -> None:
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUT_PATH.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(rows)


def _print_table(rows: list[dict[str, str]]) -> None:
    print(",".join(COLUMNS))
    for row in rows:
        print(",".join(row[column] for column in COLUMNS))
    print(f"wrote {len(rows)} rows to {OUT_PATH}")
    print("This notebook did not submit to Kaggle.")


def _score(make_labels: Callable[[np.ndarray], np.ndarray]) -> str:
    # One fast_view_counts per original annotator entry. Add the counts,
    # then read .pq once. Do not average per-image PQ. A zero denominator
    # is NaN. This notebook does not reimplement that ratio.
    total = PqCounts()
    for labels, gt_masks in _iter_views(make_labels):
        total = total + fast_view_counts(labels, gt_masks)
    return _format_pq(total.pq)


def _row(predictor: str, min_area: int, close_size: int, score: str) -> dict[str, str]:
    return {
        "predictor": predictor,
        "min_area": str(min_area),
        "close_size": str(close_size),
        "pooled_pq": score,
        "holdout_files": str(len(HOLDOUT)),
        "annotator_views": str(VIEW_COUNT),
    }


ROWS: list[dict[str, str]] = []
print(
    "Grid min_area=(80, 140, 240, 400, 700, 1200, 2000, 3000) "
    "close_size=(0, 3). Roughly log-spaced from 80 to 3000, with 400 kept. "
    "The table does not choose a winner."
)
for min_area in MIN_AREAS:
    for close_size in CLOSE_SIZES:
        print(f"baseline min_area={min_area} close_size={close_size}", flush=True)

        def _baseline(
            gray: np.ndarray,
            *,
            _area: int = min_area,
            _close: int = close_size,
        ) -> np.ndarray:
            return segment_filaments(gray, min_area=_area, close_size=_close)

        score = _score(_baseline)
        ROWS.append(_row("baseline", min_area, close_size, score))
        print(f"baseline pooled_pq={score}", flush=True)

_write_rows(ROWS)
_print_table(ROWS)

## 4. Current U-Net, upsampled to the full frame

Skipped when no `.pt` checkpoint is found. `WEIGHTS_PATH` defaults to
`None`. Set it to load that file. `None` searches every `*.pt` under
`/kaggle/working` and under `/kaggle/input`, skipping the `competitions`
tree. One file is used. Several files use `unet.pt` when exactly one
candidate has that name. Several files and no single `unet.pt` stop and
list every candidate. The loaded path is printed. No file prints a
warning and this section adds no rows, so the table stays the classic
baseline.

The network input is the training 512 area average, three channels,
ImageNet normalization, threshold 0.5. The mask is nearest-upsampled to
the grayscale frame (2048 on the train images). Instances use
`labels_from_semantic` at that size with the same grid.

On the U-Net path, nearest upsampling from 512 to 2048 turns a 1 px gap into a 4 px gap, which a 3×3 closing cannot bridge, so `close_size` 0 and 3 match on every U-Net row, while the classic baseline is labeled at full resolution and those rows do change.

`segmentation-models-pytorch` 0.6.0 is installed only in this cell, and
only when the weight file exists. The checkpoint `state_dict` is loaded
into a new `smp.Unet`. `encoder_weights` is `None` here because that
`state_dict` already contains the encoder. No Hugging Face token is set,
and this cell does not download encoder weights.

Internet has to be on for the install when the weight file is present.
Kaggle already has PyTorch. This cell does not install PyTorch and does
not submit to Kaggle.


In [ ]:
import hashlib
import random
import subprocess

from filament.instances import labels_from_semantic
from filament.train_data import downscale_pair


def _stored(checkpoint: dict, key: str) -> str:
    if key not in checkpoint or checkpoint[key] is None:
        return "not stored"
    return str(checkpoint[key])


def _tensor_sha256(state_dict: dict) -> str:
    """Hash tensor bytes so a re-zip of the same weights matches."""
    digest = hashlib.sha256()
    for key in sorted(state_dict):
        value = state_dict[key]
        digest.update(str(key).encode())
        digest.update(b"\0")
        if hasattr(value, "detach"):
            tensor = value.detach().cpu().contiguous()
            digest.update(str(tuple(tensor.shape)).encode())
            digest.update(b"\0")
            digest.update(str(tensor.dtype).encode())
            digest.update(b"\0")
            digest.update(tensor.numpy().tobytes())
        else:
            digest.update(repr(value).encode())
        digest.update(b"\0")
    return digest.hexdigest()


def _load_checkpoint_dict(path: Path) -> tuple[dict | None, str | None]:
    try:
        try:
            loaded = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            loaded = torch.load(path, map_location="cpu")
    except Exception as exc:  # noqa: BLE001
        return None, str(exc)
    if isinstance(loaded, dict):
        return loaded, None
    return None, None


def _print_fingerprint(
    label: str,
    path: Path,
    checkpoint: dict | None,
    error: str | None = None,
) -> None:
    data = path.read_bytes()
    print(f"fingerprint {label}")
    print(f"  path={path}")
    print(f"  bytes={len(data)}")
    print(f"  file_md5={hashlib.md5(data).hexdigest()}")
    if error is not None:
        print(f"  error={error}")
    if not isinstance(checkpoint, dict):
        print("  epochs=not stored")
        print("  encoder=not stored")
        print("  image_size=not stored")
        print("  tensor_sha256=not stored")
        return
    print(f"  epochs={_stored(checkpoint, 'epochs')}")
    print(f"  encoder={_stored(checkpoint, 'encoder_name')}")
    print(f"  image_size={_stored(checkpoint, 'side')}")
    state = checkpoint.get("state_dict")
    if isinstance(state, dict):
        print(f"  tensor_sha256={_tensor_sha256(state)}")
    else:
        print("  tensor_sha256=not stored")


def _print_fingerprint_note() -> None:
    print(
        "File MD5 of a re-zip is not comparable to the original file MD5. "
        "Compare tensor_sha256."
    )


def _same_file(left: Path | None, right: Path | None) -> bool:
    if left is None or right is None:
        return False
    try:
        return left.resolve() == right.resolve()
    except OSError:
        return False


def _print_other_fingerprints() -> None:
    """Fingerprint every other input checkpoint. Do not stop the sweep."""
    index = 0
    for path in _input_checkpoints():
        if _same_file(path, WEIGHTS_SOURCE) or _same_file(path, WEIGHTS_PATH):
            continue
        index += 1
        try:
            if path.is_dir():
                file_path = _rezip_extracted_archive(
                    path,
                    dest=KAGGLE_WORKING / f"fingerprint-other-{index}.pt",
                )
            else:
                file_path = path
            checkpoint, error = _load_checkpoint_dict(file_path)
            _print_fingerprint("other", file_path, checkpoint, error)
        except Exception as exc:  # noqa: BLE001
            print("fingerprint other")
            print(f"  path={path}")
            print(f"  error={exc}")
            continue


if WEIGHTS_PATH is None:
    print(
        "WARNING: no U-Net weights were loaded. "
        "Only the classic baseline is scored. "
        f"The table is in {OUT_PATH}."
    )
    try:
        import torch
    except ImportError:
        print(
            "WARNING: torch is not installed. "
            "Other checkpoint fingerprints were not printed."
        )
    else:
        try:
            _print_fingerprint_note()
            _print_other_fingerprints()
        except Exception as exc:  # noqa: BLE001
            print(f"fingerprint other error={exc}")
else:
    SMP_VERSION = "0.6.0"
    SEED = 0
    PREDICT_SIDE = 512
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            f"segmentation-models-pytorch=={SMP_VERSION}",
        ],
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"segmentation-models-pytorch=={SMP_VERSION} did not install. "
            "The baseline table is already written. No Hugging Face token "
            "was set."
        )

    import segmentation_models_pytorch as smp
    import torch

    if smp.__version__ != SMP_VERSION:
        raise RuntimeError(
            f"Expected segmentation-models-pytorch {SMP_VERSION}, "
            f"found {smp.__version__}."
        )

    def _set_seed(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    def _load_checkpoint(path: Path) -> dict:
        try:
            checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            checkpoint = torch.load(path, map_location="cpu")
        if not isinstance(checkpoint, dict) or "state_dict" not in checkpoint:
            raise RuntimeError(
                f"{path} is not the checkpoint from notebooks/train_unet.ipynb."
            )
        if checkpoint.get("encoder_name", "resnet18") != "resnet18":
            raise RuntimeError(f"{path} is not resnet18.")
        if checkpoint.get("encoder_weights", "imagenet") != "imagenet":
            raise RuntimeError(f"{path} does not record imagenet encoder weights.")
        if int(checkpoint.get("side", PREDICT_SIDE)) != PREDICT_SIDE:
            raise RuntimeError(f"{path} was not trained at side {PREDICT_SIDE}.")
        saved_smp = checkpoint.get("smp_version")
        if saved_smp is not None and saved_smp != smp.__version__:
            raise RuntimeError(
                f"{path} was saved with segmentation-models-pytorch {saved_smp}."
            )
        return checkpoint

    _set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device.type}")
    checkpoint = _load_checkpoint(WEIGHTS_PATH)
    # encoder_weights=None: the state_dict already has the encoder.
    # No Hugging Face token is set. No encoder download.
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights=None,
        in_channels=3,
        classes=1,
    )
    model.load_state_dict(checkpoint["state_dict"])
    model = model.to(device)
    model.eval()
    print(f"loaded weights={WEIGHTS_PATH} side={PREDICT_SIDE}")
    try:
        _print_fingerprint_note()
        _print_fingerprint("loaded", WEIGHTS_PATH, checkpoint)
    except Exception as exc:  # noqa: BLE001
        print(f"fingerprint loaded error={exc}")
    try:
        _print_other_fingerprints()
    except Exception as exc:  # noqa: BLE001
        print(f"fingerprint other error={exc}")

    imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

    def _semantic_full(gray: np.ndarray) -> np.ndarray:
        blank = np.zeros(gray.shape, dtype=np.uint8)
        small, _mask = downscale_pair(gray, blank, side=PREDICT_SIDE)
        scaled = small.astype(np.float32) / 255.0
        channels = np.stack((scaled, scaled, scaled), axis=0)
        normalized = (channels - imagenet_mean) / imagenet_std
        batch = torch.from_numpy(np.ascontiguousarray(normalized))
        batch = batch.unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(batch)
            mask = (torch.sigmoid(logits)[0, 0] >= 0.5).to(dtype=torch.uint8)
        small_mask = mask.detach().cpu().numpy()
        full = cv2.resize(
            small_mask,
            (gray.shape[1], gray.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        )
        return np.ascontiguousarray(full)

    for min_area in MIN_AREAS:
        for close_size in CLOSE_SIZES:
            print(f"unet min_area={min_area} close_size={close_size}", flush=True)

            def _unet(
                gray: np.ndarray,
                *,
                _area: int = min_area,
                _close: int = close_size,
            ) -> np.ndarray:
                semantic = _semantic_full(gray)
                return labels_from_semantic(
                    semantic,
                    gray,
                    min_area=_area,
                    close_size=_close,
                )

            score = _score(_unet)
            ROWS.append(_row("unet", min_area, close_size, score))
            print(f"unet pooled_pq={score}", flush=True)

    _write_rows(ROWS)
    _print_table(ROWS)